# WILD-AI plant model candidate (Google Colab)

Run in a **hosted GPU runtime**. Upload `wildai_dataset.zip` described in `training/DATASET.md`. All training runs in Colab; only exported candidate model files are downloaded for later review. This notebook does not approve a model for field use.


In [ ]:
from google.colab import files
from pathlib import Path
import csv, json, zipfile
from collections import Counter, defaultdict

uploaded = files.upload()
assert 'wildai_dataset.zip' in uploaded, 'Upload wildai_dataset.zip'
root = Path('/content/wildai_data').resolve()
root.mkdir(exist_ok=True)
with zipfile.ZipFile('/content/wildai_dataset.zip') as archive:
    for member in archive.infolist():
        destination = (root / member.filename).resolve()
        assert destination == root or root in destination.parents, 'Unsafe ZIP path'
    archive.extractall(root)

required = {'path', 'label', 'observation_id', 'split', 'source_url', 'license', 'region'}
with (root / 'manifest.csv').open(newline='', encoding='utf-8') as handle:
    reader = csv.DictReader(handle)
    assert reader.fieldnames and required <= set(reader.fieldnames), 'Manifest columns missing'
    rows = list(reader)
assert rows, 'Manifest is empty'
assert len({row['path'] for row in rows}) == len(rows), 'Duplicate image paths'
for row in rows:
    assert all(row[column].strip() for column in required), 'Blank manifest value'
    assert row['split'] in {'train', 'val', 'test'}, 'Invalid split'
    image = (root / row['path']).resolve()
    assert root in image.parents and image.is_file(), 'Missing or unsafe image path'
    row['absolute_path'] = str(image)

labels = sorted({row['label'] for row in rows} - {'__unknown__'})
assert 20 <= len(labels) <= 30, 'Need 20–30 reviewed species'
by_observation = defaultdict(set)
for row in rows:
    by_observation[row['observation_id']].add(row['split'])
assert all(len(splits) == 1 for splits in by_observation.values()), 'Observation leaks across splits'
counts = Counter((row['split'], row['label']) for row in rows)
for label in labels:
    assert all(counts[split, label] > 0 for split in ('train', 'val', 'test')), f'Missing split for {label}'
assert counts['train', '__unknown__'] == 0, 'Unknowns belong only in validation and test'
assert counts['val', '__unknown__'] >= 20 and counts['test', '__unknown__'] >= 20, 'Need more unknown examples'
print('Species:', len(labels), 'Photos:', len(rows), 'Split counts:', Counter(row['split'] for row in rows))


In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report

assert tf.config.list_physical_devices('GPU'), 'Select a hosted GPU runtime in Colab'
tf.keras.utils.set_random_seed(42)
size = (224, 224)
label_index = {label: index for index, label in enumerate(labels)}

def dataset(split, shuffle=False):
    selected = [row for row in rows if row['split'] == split and row['label'] != '__unknown__']
    paths = [row['absolute_path'] for row in selected]
    targets = [label_index[row['label']] for row in selected]
    ds = tf.data.Dataset.from_tensor_slices((paths, targets))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=42, reshuffle_each_iteration=True)
    def load(path, target):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        return tf.image.resize(tf.cast(image, tf.float32), size), target
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

def images_for(split):
    selected = [row for row in rows if row['split'] == split]
    paths = [row['absolute_path'] for row in selected]
    targets = [row['label'] for row in selected]
    ds = tf.data.Dataset.from_tensor_slices(paths)
    def load(path):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        return tf.image.resize(tf.cast(image, tf.float32), size)
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE), targets

base = tf.keras.applications.MobileNetV2(input_shape=(*size, 3), include_top=False, weights='imagenet')
base.trainable = False
inputs = tf.keras.Input(shape=(*size, 3), name='rgb_0_255')
x = tf.keras.layers.RandomFlip('horizontal')(inputs)
x = tf.keras.layers.RandomRotation(0.1)(x)
x = tf.keras.layers.Rescaling(1 / 127.5, offset=-1)(x)
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(len(labels), activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(dataset('train', shuffle=True), validation_data=dataset('val'), epochs=20,
          callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)])

val_images, val_labels = images_for('val')
val_scores = model.predict(val_images, verbose=0)
unknown_confidence = val_scores[np.array(val_labels) == '__unknown__'].max(axis=1)
threshold = float(np.nextafter(np.quantile(unknown_confidence, 0.95, method='higher'), 1.0))

test_images, test_labels = images_for('test')
test_scores = model.predict(test_images, verbose=0)
test_confidence = test_scores.max(axis=1)
test_predictions = [labels[index] if confidence >= threshold else '__unknown__'
                    for index, confidence in zip(test_scores.argmax(axis=1), test_confidence)]
print(classification_report(test_labels, test_predictions, labels=labels + ['__unknown__'], zero_division=0))
known_indices = [i for i, label in enumerate(test_labels) if label != '__unknown__']
unknown_indices = [i for i, label in enumerate(test_labels) if label == '__unknown__']
metrics = {
    'known_correct_rate': sum(test_predictions[i] == test_labels[i] for i in known_indices) / len(known_indices),
    'known_abstention_rate': sum(test_predictions[i] == '__unknown__' for i in known_indices) / len(known_indices),
    'unknown_false_accept_rate': sum(test_predictions[i] != '__unknown__' for i in unknown_indices) / len(unknown_indices),
    'test_images': len(test_labels),
    'threshold': threshold,
}
print(metrics)

output = Path('/content/wildai_model_candidate')
output.mkdir(exist_ok=True)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
(output / 'plant_model.tflite').write_bytes(converter.convert())
(output / 'labels.json').write_text(json.dumps({'labels': labels, 'image_size': list(size),
    'input': 'float32 RGB 0..255', 'abstain_below': threshold, 'status': 'candidate_only'}, indent=2))
(output / 'evaluation.json').write_text(json.dumps({'metrics': metrics, 'tensorflow': tf.__version__,
    'counts': {f'{split}:{label}': count for (split, label), count in counts.items()}}, indent=2))
with zipfile.ZipFile('/content/wildai_model_candidate.zip', 'w') as archive:
    for path in output.iterdir():
        archive.write(path, path.name)
files.download('/content/wildai_model_candidate.zip')
